
# 3-Qubit Repetition-Code QEC on Quantinuum Nexus

This notebook combines the main workflow of the two original notebooks into one continuous story:

$$
\boxed{\text{Build QEC circuit}}
\rightarrow
\boxed{\text{Ideal emulator}}
\rightarrow
\boxed{\text{Noisy emulator}}
\rightarrow
\boxed{\text{No-QEC baseline}}
$$

The code is the 3-qubit repetition code for correcting a single $X$ (bit-flip) error:

$$
|0_L\rangle = |000\rangle .
$$

We deliberately inject an error on the middle data qubit,

$$
|000\rangle \xrightarrow{X_1} |010\rangle ,
$$

measure two parity checks,

$$
Z_0Z_1,\qquad Z_1Z_2,
$$

decode the syndrome, and apply a classically conditioned correction.

The expected ideal flow is

$$
|000\rangle
\rightarrow |010\rangle
\rightarrow (s_0,s_1)=(1,1)
\rightarrow X_1
\rightarrow |000\rangle .
$$

The notebook then repeats the same experiment with the H2 emulator noise model enabled, and finally keeps the original no-QEC baseline for comparison.



## 1. Syndrome logic

The two measured syndrome bits are

$$
s_0=d_0\oplus d_1,\qquad
s_1=d_1\oplus d_2.
$$

For a single $X$ error:

| Error | $(s_0,s_1)$ | Register integer $s_0+2s_1$ | Correction |
|---|---:|---:|---|
| none | (0,0) | 0 | none |
| $X_0$ | (1,0) | 1 | $X_0$ |
| $X_1$ | (1,1) | 3 | $X_1$ |
| $X_2$ | (0,1) | 2 | $X_2$ |

The decoder in these notebooks is therefore a simple real-time lookup implemented with classical conditions:

```python
reg_eq(syndrome, 1)  -> X(data[0])
reg_eq(syndrome, 3)  -> X(data[1])
reg_eq(syndrome, 2)  -> X(data[2])
```

This is the same basic QEC loop in both original notebooks; only the emulator noise setting changes.


In [4]:

# ============================================================
# 2. Imports and Nexus project
# ============================================================

import qnexus as qnx
import uuid

from pytket.circuit import Circuit
from pytket.circuit.logic_exp import reg_eq

suffix = str(uuid.uuid4())[:8]

project = qnx.projects.get_or_create(
    name="QEC Repetition Code Walkthrough"
)

qnx.context.set_active_project(project)

print("Project ready")
print("suffix =", suffix)


Project ready
suffix = 4c502bd3



## 3. Build the QEC circuit once

We use:

- 3 data qubits: `data[0:3]`
- 1 ancilla qubit reused for both parity measurements
- 2 classical syndrome bits
- 3 final output bits

The ancilla is reset after each mid-circuit measurement so it can be reused.

The same circuit will be run twice:

1. with `noisy_simulation=False`;
2. with `noisy_simulation=True`.


In [5]:

# ============================================================
# 3. Build the 3-qubit repetition-code QEC circuit
# ============================================================

circuit = Circuit()
circuit.name = "3-qubit repetition QEC"

# 3 data qubits
data = circuit.add_q_register("data", 3)

# 1 ancilla reused for syndrome extraction
ancilla = circuit.add_q_register("ancilla", 1)

# 2 syndrome bits
syndrome = circuit.add_c_register("syndrome", 2)

# 3 final measurement bits
output = circuit.add_c_register("output", 3)


# ------------------------------------------------------------
# Step A: logical |0_L> = |000>
# ------------------------------------------------------------
# Qubits start automatically in |0>, so no preparation gates
# are needed for |000>.


# ------------------------------------------------------------
# Step B: deliberately inject X on the middle qubit
#
# |000> -> |010>
# ------------------------------------------------------------

circuit.X(data[1])


# ------------------------------------------------------------
# Step C: measure Z0 Z1
#
# syndrome[0] = data[0] XOR data[1]
# ------------------------------------------------------------

circuit.CX(data[0], ancilla[0])
circuit.CX(data[1], ancilla[0])

circuit.Measure(
    ancilla[0],
    syndrome[0]
)

circuit.Reset(
    ancilla[0]
)


# ------------------------------------------------------------
# Step D: measure Z1 Z2
#
# syndrome[1] = data[1] XOR data[2]
# ------------------------------------------------------------

circuit.CX(data[1], ancilla[0])
circuit.CX(data[2], ancilla[0])

circuit.Measure(
    ancilla[0],
    syndrome[1]
)

circuit.Reset(
    ancilla[0]
)


# ------------------------------------------------------------
# Step E: classical decoder + conditional correction
#
# (s0,s1) = (1,0) -> integer 1 -> correct data[0]
# (s0,s1) = (1,1) -> integer 3 -> correct data[1]
# (s0,s1) = (0,1) -> integer 2 -> correct data[2]
# ------------------------------------------------------------

circuit.X(
    data[0],
    condition=reg_eq(syndrome, 1)
)

circuit.X(
    data[1],
    condition=reg_eq(syndrome, 3)
)

circuit.X(
    data[2],
    condition=reg_eq(syndrome, 2)
)


# ------------------------------------------------------------
# Step F: final measurement of the 3 data qubits
# ------------------------------------------------------------

for i in range(3):
    circuit.Measure(
        data[i],
        output[i]
    )

print(circuit)


<tket::Circuit, qubits=4, gates=18>



## 4. Helper: upload → compile → execute → download

Both experiments use the same Nexus workflow:

$$
\text{Circuit}
\rightarrow
\text{upload}
\rightarrow
\text{compile for H2-Emulator}
\rightarrow
\text{execute}
\rightarrow
\text{download result}.
$$

Keeping this in one helper function makes the difference between the ideal and noisy runs very explicit: the circuit is unchanged; only `noisy_simulation` and the number of shots change.


In [6]:

# ============================================================
# 4. Helper function for a complete Nexus emulator run
# ============================================================

def run_on_h2_emulator(
    circuit,
    *,
    noisy_simulation,
    n_shots,
    label,
):
    config = qnx.QuantinuumConfig(
        device_name="H2-Emulator",
        noisy_simulation=noisy_simulation
    )

    circuit_ref = qnx.circuits.upload(
        circuit=circuit,
        name=f"{label}-circuit-{suffix}"
    )

    print(f"[{label}] Circuit uploaded")

    compile_job = qnx.start_compile_job(
        programs=[circuit_ref],
        backend_config=config,
        optimisation_level=0,
        name=f"{label}-compile-{suffix}"
    )

    print(f"[{label}] Compilation submitted")
    qnx.jobs.wait_for(compile_job)

    print(
        f"[{label}] Compilation status:",
        qnx.jobs.status(compile_job)
    )

    compiled_ref = qnx.jobs.results(
        compile_job
    )[0].get_output()

    execute_job = qnx.start_execute_job(
        programs=[compiled_ref],
        backend_config=config,
        n_shots=[n_shots],
        name=f"{label}-execute-{suffix}"
    )

    print(f"[{label}] Execution submitted")
    qnx.jobs.wait_for(execute_job)

    print(
        f"[{label}] Execution status:",
        qnx.jobs.status(execute_job)
    )

    result = qnx.jobs.results(
        execute_job
    )[0].download_result()

    print(f"[{label}] Result downloaded")

    return result



# Part I — Ideal QEC

First switch the H2 emulator noise model **off**:

```python
noisy_simulation=False
```

This isolates the QEC logic itself.

Because we deliberately inserted $X_1$, the ideal result should be:

$$
(s_0,s_1)=(1,1)
$$

for every shot, followed by correction back to

$$
|000\rangle.
$$

The original ideal notebook used 100 shots, so we keep that here.


In [7]:

# ============================================================
# 5. IDEAL experiment
# ============================================================

ideal_shots = 100

ideal_result = run_on_h2_emulator(
    circuit,
    noisy_simulation=False,
    n_shots=ideal_shots,
    label="ideal-qec",
)

ideal_syndrome_counts = ideal_result.get_counts(
    [syndrome[0], syndrome[1]]
)

ideal_output_counts = ideal_result.get_counts(
    [output[0], output[1], output[2]]
)

print()
print("=== IDEAL: SYNDROME ===")
print(ideal_syndrome_counts)

print()
print("=== IDEAL: FINAL DATA ===")
print(ideal_output_counts)

ideal_success = ideal_output_counts.get(
    (0, 0, 0),
    0
)

print()
print(
    "Ideal exact recovery rate =",
    ideal_success / ideal_shots
)


[ideal-qec] Circuit uploaded
[ideal-qec] Compilation submitted
[ideal-qec] Compilation status: JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 9, 26, 13, 19, 5, 189778, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 9, 26, 13, 19, 2, 377852, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 9, 26, 13, 18, 56, 14787, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 9, 26, 13, 19, 2, 827647, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)
[ideal-qec] Execution submitted
[ideal-qec] Execution status: JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 9, 26, 13, 20, 41, 750426, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 9, 26, 13, 19, 35, 713156, tzinfo=datetime.


# Part II — QEC with the H2 emulator noise model

Now run **the same QEC circuit** with

```python
noisy_simulation=True
```

and increase to 1000 shots, matching the noisy notebook.

The logical structure is unchanged:

$$
|000\rangle
\rightarrow X_1
\rightarrow \text{syndrome extraction}
\rightarrow \text{conditional correction}
\rightarrow \text{final measurement}.
$$

The difference is that the emulator now introduces noise during the execution, so the syndrome and final output no longer have to be perfect on every shot.


In [8]:

# ============================================================
# 6. NOISY experiment
# ============================================================

noisy_shots = 1000

noisy_result = run_on_h2_emulator(
    circuit,
    noisy_simulation=True,
    n_shots=noisy_shots,
    label="noisy-qec",
)

noisy_syndrome_counts = noisy_result.get_counts(
    [syndrome[0], syndrome[1]]
)

noisy_output_counts = noisy_result.get_counts(
    [output[0], output[1], output[2]]
)

print()
print("=== NOISY QEC: SYNDROME ===")
print(noisy_syndrome_counts)

print()
print("=== NOISY QEC: FINAL DATA ===")
print(noisy_output_counts)

noisy_success = noisy_output_counts.get(
    (0, 0, 0),
    0
)

noisy_success_rate = noisy_success / noisy_shots
noisy_failure_rate = 1.0 - noisy_success_rate

print()
print("Successful exact recoveries =", noisy_success)
print("QEC success rate =", noisy_success_rate)
print("QEC failure rate =", noisy_failure_rate)


[noisy-qec] Circuit uploaded
[noisy-qec] Compilation submitted
[noisy-qec] Compilation status: JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 9, 26, 13, 20, 54, 929966, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 9, 26, 13, 20, 52, 419300, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 9, 26, 13, 20, 45, 136351, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 9, 26, 13, 20, 53, 143172, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)
[noisy-qec] Execution submitted
[noisy-qec] Execution status: JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 9, 26, 13, 22, 20, 611356, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 9, 26, 13, 21, 5, 594424, tzinfo=dateti


# Part III — No-QEC baseline

The baseline from the noisy notebook is kept.

It uses the same initial state and the same deliberately injected middle-qubit error:

$$
|000\rangle \xrightarrow{X_1} |010\rangle .
$$

But it removes all QEC operations:

- no syndrome extraction;
- no decoder;
- no correction.

The three data qubits are measured directly.

This baseline therefore answers a simple functionality question:

> If the known $X_1$ error is present, what happens when we do not run the QEC loop?


In [9]:

# ============================================================
# 7. Build the no-QEC baseline
# ============================================================

baseline = Circuit()
baseline.name = "No QEC baseline"

data_b = baseline.add_q_register(
    "data",
    3
)

output_b = baseline.add_c_register(
    "output",
    3
)

# Start in |000>
#
# Inject exactly the same error as the QEC experiment:
# |000> -> |010>
baseline.X(
    data_b[1]
)

# No syndrome extraction
# No decoder
# No correction

for i in range(3):
    baseline.Measure(
        data_b[i],
        output_b[i]
    )

print(baseline)


<tket::Circuit, qubits=3, gates=4>


In [10]:

# ============================================================
# 8. Run the baseline with the same noisy H2 emulator
# ============================================================

baseline_result = run_on_h2_emulator(
    baseline,
    noisy_simulation=True,
    n_shots=noisy_shots,
    label="no-qec-baseline",
)

baseline_counts = baseline_result.get_counts(
    [output_b[0], output_b[1], output_b[2]]
)

print()
print("=== NO-QEC BASELINE: FINAL DATA ===")
print(baseline_counts)

baseline_success = baseline_counts.get(
    (0, 0, 0),
    0
)

baseline_success_rate = baseline_success / noisy_shots
baseline_failure_rate = 1.0 - baseline_success_rate

print()
print("Baseline success rate =", baseline_success_rate)
print("Baseline failure rate =", baseline_failure_rate)

print()
print("QEC success rate =", noisy_success_rate)
print("QEC failure rate =", noisy_failure_rate)


[no-qec-baseline] Circuit uploaded
[no-qec-baseline] Compilation submitted
[no-qec-baseline] Compilation status: JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 9, 26, 13, 22, 33, 469696, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 9, 26, 13, 22, 32, 356497, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 9, 26, 13, 22, 22, 663670, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 9, 26, 13, 22, 32, 754964, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)
[no-qec-baseline] Execution submitted
[no-qec-baseline] Execution status: JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 9, 26, 13, 23, 7, 688095, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 9, 26, 13,


# 9. Main takeaway

The notebook has now followed one continuous experimental axis:

### A. Verify the QEC logic

$$
\text{H2-Emulator, noise off}
$$

Expected:

$$
11\rightarrow X_1\rightarrow000.
$$

### B. Turn on emulator noise

$$
\text{same QEC circuit}
+
\text{H2 noise model}.
$$

Now the syndrome and final output can contain errors, so the experiment reports an empirical exact-recovery rate.

### C. Compare with the retained no-QEC baseline

The baseline deliberately receives the same $X_1$ error but performs no syndrome extraction and no correction.

So the overall workflow is:

$$
\boxed{
\text{ideal functionality check}
\rightarrow
\text{noisy QEC run}
\rightarrow
\text{no-QEC baseline}
}
$$

This notebook intentionally keeps the decoder simple:

$$
\text{syndrome}
\rightarrow
\text{classical condition}
\rightarrow
X_i.
$$

It does **not** use a Wasm decoder; the purpose here is to make the complete repetition-code QEC control flow easy to see from beginning to end.
